# Books Banned in Texas State Prisons

In [1]:
# step 1: import pandas
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import string

import requests # used for webscraping
import time # used for webscraping

import re # to better manipulate regular expressions

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# the 'magic' or the percent symbol means it will look good visually

In [2]:
# step 2: import .csv file
data = pd.read_csv("data/banned_book_data_tx-list.csv")

data.head(10)

,publication,author,date,unit_deny_reason,reason,"note_on_sourcing:_i_got_this_information_through_a_request_to_the_tdcj_public_information_office,_received_10/14/2022.",year,month,day,state_arc
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,NaN,2016,8,19,tx
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,NaN,2012,9,10,tx
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,NaN,2019,7,17,tx
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,NaN,2010,1,29,tx
4,A BLACK GAZE,"CAMPT, TINA",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",NaN,2021,9,8,tx
5,A BLACK ROSE THRIVED,"RICHEY, ROCHELLE",2018-08-15,AGE 33 CONTAINS THE GRAPHIC DEPICTION OF RAPE ...,PG 33 RAPE OF MINOR (AGE ON PAGE 32),NaN,2018,8,15,tx
6,A BRIEF HISTORY OF ALBUM COVERS,"DRAPER, JASON",2020-12-31,"PAGES 51 & 245 NUDE CHILD; PAGES 81, 115, 141 ...",PAGES 5 & 245 NUDE CHILD,NaN,2020,12,31,tx
7,A BRIEF HISTORY OF MANGA,"MCCARTHY, HELEN",2014-10-15,PAGE 33 CONTAINS SEXUALLY EXPLICIT IMAGE,PG 33 SEXUALLY EXPLICIT IMAGE,NaN,2014,10,15,tx
8,A BRIEF HISTORY OF VICE,"EVANS, ROBERT",2016-09-23,AGES 212 AND 213 CONTAIN SEXUALLY EXPLICIT IMAGES,PG 212 CONTAINS A SEXUALLY EXPLICIT IMAGE,NaN,2016,9,23,tx
9,A CELEBRATION OF CHILDHOOD,"SUDELL, HELEN",2015-01-21,G.46 CONTAINS A PHOTO OF A NUDE CHILD,PG 46 NUDE CHILD,NaN,2015,1,21,tx


# Note on Sourcing (Webscraping)
## Open Library API documentation
### Open Library's API documentation can be found at this link: https://openlibrary.org/dev/docs/api/search

### I used python's library documentation as a reference: https://docs.python.org/3.4/library/stdtypes.html

### With regular expression documentation here: https://docs.python.org/3.4/library/re.html#module-re

## Library of Congress API documentation
### can be found here: https://www.loc.gov/apis/json-and-yaml/requests/

In [3]:
# test the webscraping on a single title
title = "When Breath Becomes Air"

# using formatted string literals 'f-strings' to take a string and a variable
# in the same statement
# variables are denoted by {}

# step 1: make an HTTP request to the API endpoint
url = f"https://openlibrary.org/search.json?title={title}"

# step 2: parse the JSON response
response = requests.get(url)
new_data = response.json()

# print(new_data.keys())

# step 3: extract and print the new data
# if the docs key does not contain subject, return []
print(new_data["docs"][0]["title"])
print(new_data["docs"][0].get("subject", []))

When Breath Becomes Air
[]


In [4]:
# print(new_data.keys())
# docs = new_data.get("docs", [])
# if docs:
    # print(docs[0].keys())

work_key = new_data["docs"][0]["key"]

work = requests.get(
    f"https://openlibrary.org{work_key}.json"
).json()

# print(work.keys())
print(work.get("description"))

When Breath Becomes Air is a non-fiction autobiographical book written by American neurosurgeon Paul Kalanithi. It is a memoir about his life and illness, battling stage IV metastatic lung cancer. It was posthumously published by Random House on January 12, 2016.


## Inspect the Data

In [5]:
data.shape

(9396, 10)

In [6]:
data.columns

Index(['publication', 'author', 'date', 'unit_deny_reason', 'reason',
       'note_on_sourcing:_i_got_this_information_through_a_request_to_the_tdcj_public_information_office,_received_10/14/2022.',
       'year', 'month', 'day', 'state_arc'],
      dtype='object')

## Clean up Column Names

In [7]:
data = data.rename(columns={
    "publication": "publication",
    "author": "author",
    "date": "date",
    "unit_deny_reason": "unit_reason",
    "reason": "reason",
    "year": "year",
    "month": "month",
    "day": "day",
    "state_arc": "state"
})

data = data.drop(columns=["note_on_sourcing:_i_got_this_information_through_a_request_to_the_tdcj_public_information_office,_received_10/14/2022."], errors="ignore")
data.head()

,publication,author,date,unit_reason,reason,year,month,day,state
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,2016,8,19,tx
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,2012,9,10,tx
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,2019,7,17,tx
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,2010,1,29,tx
4,A BLACK GAZE,"CAMPT, TINA",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",2021,9,8,tx


## Reinspect the Data

In [8]:
data.shape

(9396, 9)

In [9]:
data.columns

Index(['publication', 'author', 'date', 'unit_reason', 'reason', 'year',
       'month', 'day', 'state'],
      dtype='object')

# Clean the Data

In [10]:
# clean the data
## step 1: convert to lowercase
data["clean_reason"] = (
    data["reason"]
    .str.lower()
)

data["clean_reason"].head()

0                 pgs 81 & 369 sexually explicit image
1                                    pg 422 nude child
2                       pg 294 sexually explicit image
3    page 9 of photo insert contains sexually expli...
4    pages 28, 30 & 35 contain sexually explicit im...
Name: clean_reason, dtype: object

In [11]:
# clean the data
## step 2: remove punctuation
data["clean_reason"] = (
    data["clean_reason"]
    .str.translate(
        str.maketrans("", "", string.punctuation)
    )
)

data["clean_reason"].head()

0                  pgs 81  369 sexually explicit image
1                                    pg 422 nude child
2                       pg 294 sexually explicit image
3    page 9 of photo insert contains sexually expli...
4     pages 28 30  35 contain sexually explicit images
Name: clean_reason, dtype: object

In [12]:
# clean the data
## step 3: remove 'pgs' and numbers
data["clean_reason"] = (
    data["clean_reason"]
    .str.replace(r"\bpgs\b", "", regex = True) # remove pgs
    .str.replace(r"\bpg\b", "", regex = True) # remove pg
    .str.replace(r"\bpage\b", "", regex = True) # remove page
    .str.replace(r"\bpages\b", "", regex = True) # remove pages
    .str.replace(r"\d+", "", regex = True) # remove numbers
    .str.strip()
)

data["clean_reason"].head(100)

0                               sexually explicit image
1                                            nude child
2                               sexually explicit image
3     of photo insert contains sexually explicit images
4                      contain sexually explicit images
                            ...                        
95                                     sex with a minor
96                                incest brotherbrother
97                                                 rape
98                                                 rape
99                                                 rape
Name: clean_reason, Length: 100, dtype: object

In [13]:
# clean the data
## step 4: remove stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)
# import english stop words and make them a set called stopwords

In [14]:
data["clean_reason"] = data["clean_reason"].apply( # have column
    lambda x: " ".join(
        word for word in x.split()
        if word not in stopwords
    ) if isinstance(x,str) else "" # if data is not string, replace with string
)

data["clean_reason"].head(10)

0                           sexually explicit image
1                                        nude child
2                           sexually explicit image
3    photo insert contains sexually explicit images
4                  contain sexually explicit images
5                                    rape minor age
6                                        nude child
7                           sexually explicit image
8                  contains sexually explicit image
9                                        nude child
Name: clean_reason, dtype: object

In [15]:
print(len(data)) # there are 9396 entries...

9396


In [16]:
print(data["publication"].nunique()) # but only 9119 unique publications

9119


In [17]:
data[["publication", "author"]].head(10)

,publication,author
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT"
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA"
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID"
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID"
4,A BLACK GAZE,"CAMPT, TINA"
5,A BLACK ROSE THRIVED,"RICHEY, ROCHELLE"
6,A BRIEF HISTORY OF ALBUM COVERS,"DRAPER, JASON"
7,A BRIEF HISTORY OF MANGA,"MCCARTHY, HELEN"
8,A BRIEF HISTORY OF VICE,"EVANS, ROBERT"
9,A CELEBRATION OF CHILDHOOD,"SUDELL, HELEN"


In [18]:
# clean the data: title format
# lets define a function 'clean_title'
def clean_title(title):
    if pd.isna(title):
        return None

    title = str(title).strip()

    # re.sub(pattern, repl, string, count=0, flags=0)
    # return the string obtained by replacing the left-most non-overlapping
        # occurences of pattern in string by the replacement repl
            # r = raw string
            # , = ,
            # $ = if at the end of the string
        # en gros: replace trailing commas with nothing
    title = re.sub(r',$', '', title)

    # collapse spaces (solves problem of accidental double spacing)
    # \s = any whitespace character i.e. " " "\t" "\n"
    # + = one or more in a row
    title = re.sub(r'\s+', ' ', title)

    return title

In [19]:
# clean the data: author format
# let's define a function 'author_first_last'
# the .strip() function removes leading and trailing spaces/characters
    #(depending on the argument) to return a copy of the original string

def author_first_last(author):
    
    if pd.isna(author):
        return None

    author = author.strip() # default is spaces

    if "," in author:
        last, first = author.split(",", 1) # using , as the delimiter
        return f"{first.strip()} {last.strip()}" # recombine first & last name

    return author

In [20]:
data["clean_author"] = (
    data["author"]
    .apply(author_first_last)
    .str.title()
)

In [21]:
data["clean_author"].head()

0       Robert Howard
1    Barbara Thiemann
2     David Henderson
3     David Henderson
4          Tina Campt
Name: clean_author, dtype: object

In [23]:
data["author"] = (
    data["author"]
    .str.lower()
)

In [24]:
data["clean_title"] = (
    data["publication"]
    .apply(clean_title)
    .str.title()
)

In [25]:
data["clean_title"].head()

0            ...And Their Memory Was A Bitter Tree
1    (Non)Conform Russian And Soviet Art 1958-1995
2                    Scuse Me While I Kiss The Sky
3                    Scuse Me While I Kiss The Sky
4                                     A Black Gaze
Name: clean_title, dtype: object

In [26]:
data.head()

,publication,author,date,unit_reason,reason,year,month,day,state,clean_reason,clean_author,clean_title
0,...AND THEIR MEMORY WAS A BITTER TREE,"howard, robert",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,2016,8,19,tx,sexually explicit image,Robert Howard,...And Their Memory Was A Bitter Tree
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","thiemann, barbara",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,2012,9,10,tx,nude child,Barbara Thiemann,(Non)Conform Russian And Soviet Art 1958-1995
2,SCUSE ME WHILE I KISS THE SKY,"henderson, david",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,2019,7,17,tx,sexually explicit image,David Henderson,Scuse Me While I Kiss The Sky
3,SCUSE ME WHILE I KISS THE SKY,"henderson, david",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,2010,1,29,tx,photo insert contains sexually explicit images,David Henderson,Scuse Me While I Kiss The Sky
4,A BLACK GAZE,"campt, tina",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",2021,9,8,tx,contain sexually explicit images,Tina Campt,A Black Gaze


## Let's Create a Unique Books List

In [138]:
books = (
    data[["clean_title", "clean_author"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

books.shape

(9262, 2)

In [139]:
# function to retrieve subjects from openlibrary.org
# takes the title of a book and the author name First Last as params
def get_subjects(title, author):

    headers = {
        "User-Agent": "SummerBookProject/1.0 (Ava Pettey; arppettey@gmail.com)"
    }
    
    try: # the try block tests a block of code for errors...

        params = {
            "title": title,
            "author": author
        }

        search = requests.get( # send a GET request to the API
            "https://openlibrary.org/search.json", # API provided URL
            params=params,
            headers=headers,
            timeout=20
        ).json()

        docs = search.get("docs", [])

        if len(docs) == 0:
            return []
            # return ["doc len 0"]

        work_key = docs[0].get("key")

        if not work_key:
            return []
            # return ["no key"]

        work = requests.get(
            f"https://openlibrary.org{work_key}.json",
            headers=headers,
            timeout=20
        ).json()

        subjects = work.get("subjects", [])

        return subjects

    except Exception: # ...the except block handles those errors

        return [] # let's return [] on error

# print(len(docs))

In [140]:
# function to cite source
def identify_source(subjects):

    text =" ".join(subjects).lower()

    if not pd.isna(text):
        return "openlibrary.org"

    return NA

In [141]:
test = books.head(20)

for index, row in test.iterrows():

    subjects = get_subjects(
        row["clean_title"],
        row["clean_author"]
    )

    print(row["clean_title"])
    print(subjects[:5]) # get the first five subjects
    print()

...And Their Memory Was A Bitter Tree
['Fiction, fantasy, short stories']

(Non)Conform Russian And Soviet Art 1958-1995
[]

Scuse Me While I Kiss The Sky
[]

A Black Gaze
[]

A Black Rose Thrived
[]

A Brief History Of Album Covers
['Sound recordings', 'Packaging']

A Brief History Of Manga
[]

A Brief History Of Vice
['Vice', 'Modern Civilization', 'History', 'Civilization, modern', 'nyt:fashion-manners-and-customs=2016-09-11']

A Celebration Of Childhood
[]

A Centaur'S Life 10
['Centaurs', 'Comic books, strips', 'High school students', 'Mythical Animals', 'Graphic novels']

A Centaur'S Life 12
[]

A Centaur'S Life 13
[]

A Centaur'S Life 14
[]

A Centaur'S Life 15
[]

A Centaur'S Life 2
['Juvenile fiction', 'Graphic novels', 'High school students', 'Centaurs', 'High schools']

A Centaur'S Life 6
[]

A Certain Scientific Accelerator V1
[]

A Child Is Born
['Pregnancy', 'Childbirth', 'Pictorialworks', 'Pictorial works', 'Pregnancy, pictorial works']

A Child Of A Crackhead
['Children

In [28]:
# what percent of the publications include volumes? about 10%
    # r = raw string
    # \b = boundary for matching words, not parts of words
    # \d+ = one or more digits "V1"
# books["clean_title"].str.contains(
    # r"\b(V\d+|VOL|VOLUME|BK\d+|BOOK\s+\d+)\b",
    # case=False,
    # regex=True
# ).mean()

In [36]:
# collect data from openlibrary.org
# books["subjects"] = None
# books["source"] = None
books = pd.read_csv("openlibrary_subjects_match.csv")

for i, row in books.iterrows():

    # skip the filled rows
    if pd.notna(row["subjects"]):
        continue

    subjects = get_subjects(
        row["clean_title"],
        row["clean_author"]
    )

    books.at[i, "subjects"] = subjects
    books.at[i, "source"] = identify_source(subjects)

    if i % 500 == 0: # save every 500 books
        books.to_csv(
            "openlibrary_subjects_match.csv",
            index=False
        )

    if i % 100 == 0:
        print(f"{i} completed")

    time.sleep(0.2)

# final save
books.to_csv(
    "openlibrary_subjects_match.csv",
    index=False
)

print("finished loading all data")

8700 completed
8800 completed
8900 completed
9000 completed
9100 completed
9200 completed
finished loading all data


In [37]:
# final save
books.to_csv("openlibrary_subjects_match.csv", index=False)
# print(books["subjects"].notna().sum())
books.head(10)

,clean_title,clean_author,subjects,source
0,...And Their Memory Was A Bitter Tree,Robert Howard,"['Fiction, fantasy, short stories']",openlibrary.org
1,(Non)Conform Russian And Soviet Art 1958-1995,Barbara Thiemann,[],openlibrary.org
2,Scuse Me While I Kiss The Sky,David Henderson,[],openlibrary.org
3,A Black Gaze,Tina Campt,[],openlibrary.org
4,A Black Rose Thrived,Rochelle Richey,[],openlibrary.org
5,A Brief History Of Album Covers,Jason Draper,"['Sound recordings', 'Packaging']",openlibrary.org
6,A Brief History Of Manga,Helen Mccarthy,[],openlibrary.org
7,A Brief History Of Vice,Robert Evans,"['Vice', 'Modern Civilization', 'History', 'Ci...",openlibrary.org
8,A Celebration Of Childhood,Helen Sudell,[],openlibrary.org
9,A Centaur'S Life 10,Kei Murayama,"['Centaurs', 'Comic books, strips', 'High scho...",openlibrary.org


In [32]:
# for those that did not get subjects, access library of congress API
# r = requests.get("https://www.loc.gov/books/?fo=json")
# books_congress = r.json()

#for book in books_congress["results"]:
    # print(book.get("title"))
    # print(book.get("subject"))
    # print(book.get("contributor"))
    # print(book.get("id"))
    
    #print(books_congress["results"][0].keys())

In [38]:
# lets create another unique books_second set
books_second = (
    data[["clean_title", "author"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

books_second.head(10)

,clean_title,author
0,...And Their Memory Was A Bitter Tree,"howard, robert"
1,(Non)Conform Russian And Soviet Art 1958-1995,"thiemann, barbara"
2,Scuse Me While I Kiss The Sky,"henderson, david"
3,A Black Gaze,"campt, tina"
4,A Black Rose Thrived,"richey, rochelle"
5,A Brief History Of Album Covers,"draper, jason"
6,A Brief History Of Manga,"mccarthy, helen"
7,A Brief History Of Vice,"evans, robert"
8,A Celebration Of Childhood,"sudell, helen"
9,A Centaur'S Life 10,"murayama, kei"


In [51]:
# for all books in the .csv file without subjects yet
# function to retrieve subject keys from library of congress
# takes the title of a book and the contributor name last, first as params
books_second = pd.read_csv("library_subjects_match.csv")

def get_subject(title, contributor):

    headers = {
        "User-Agent": "SummerBookProject/1.0 (Ava Pettey; arppettey@gmail.com)"
    }
    
    try: # the try block tests a block of code for errors...

        params = {
            "q": f'"{title}" {contributor}'
        }

        req = requests.get( # send a GET request to the API
            "https://www.loc.gov/books/?fo=json", # API provided URL
            params=params,
            headers=headers,
            timeout=20
        )

        if req.status_code == 429:
            print("Rate limited")
            time.sleep(60)
            return []

        try:
            search = req.json()
        except Exception:
            return []

        results = search.get("results", [])

        if not results:
            return []

        return results[0].get("subject", []) # request the subject of first result

    except Exception as e:
        print(f"Error for {title}: {e}")
        return []

In [ ]:
my_test = get_subject(
    row["clean_title"],
    row["recleaned_author"]
)

print(subject)

Rate limited


In [43]:
books_second.head()

,clean_title,clean_author,subjects,source,recleaned_author
0,...And Their Memory Was A Bitter Tree,Robert Howard,"['Fiction, fantasy, short stories']",openlibrary.org,"howard, robert"
1,(Non)Conform Russian And Soviet Art 1958-1995,Barbara Thiemann,[],library of congress,"thiemann, barbara"
2,Scuse Me While I Kiss The Sky,David Henderson,[],library of congress,"henderson, david"
3,A Black Gaze,Tina Campt,[],library of congress,"henderson, david"
4,A Black Rose Thrived,Rochelle Richey,[],library of congress,"campt, tina"


In [152]:
# function to cite source
def identify_second_source(subject):

    text =" ".join(subject).lower()
 
    if not pd.isna(text):
        return "library of congress"

    return NA

In [153]:
books_second.head()

,clean_title,clean_author,subjects,source,recleaned_author
0,...And Their Memory Was A Bitter Tree,Robert Howard,"['Fiction, fantasy, short stories']",openlibrary.org,"howard, robert"
1,(Non)Conform Russian And Soviet Art 1958-1995,Barbara Thiemann,[],library of congress,"thiemann, barbara"
2,Scuse Me While I Kiss The Sky,David Henderson,[],library of congress,"henderson, david"
3,A Black Gaze,Tina Campt,[],library of congress,"henderson, david"
4,A Black Rose Thrived,Rochelle Richey,[],library of congress,"campt, tina"


In [150]:
test_two = books_second.head(20)

for index, row in test_two.iterrows():

    subject = get_subject(
        row["clean_title"],
        row["recleaned_author"]
    )

    print(row["clean_title"])
    print(row["recleaned_author"])
    print(subject[:5]) # get the first five subjects
    print()

...And Their Memory Was A Bitter Tree
howard, robert
[]

(Non)Conform Russian And Soviet Art 1958-1995
thiemann, barbara
[]

Scuse Me While I Kiss The Sky
henderson, david
[]

A Black Gaze
henderson, david
[]

A Black Rose Thrived
campt, tina
[]

A Brief History Of Album Covers
richey, rochelle
[]

A Brief History Of Manga
draper, jason
[]

A Brief History Of Vice
mccarthy, helen
[]

A Celebration Of Childhood
evans, robert
[]

A Centaur'S Life 10
sudell, helen
[]

A Centaur'S Life 12
murayama, kei
[]

A Centaur'S Life 13
murayama, kei
[]

A Centaur'S Life 14
murayama, kei
[]

A Centaur'S Life 15
murayama, kei
[]

A Centaur'S Life 2
murayama, kei
[]

A Centaur'S Life 6
murayama, kei
[]

A Certain Scientific Accelerator V1
murayama, kei
[]

A Child Is Born
kamachi, kazuma
[]

A Child Of A Crackhead
nilsson, lennart
[]

A Child Of A Crackhead 2 A Fight To Stay Alive
speight, shameek
[]



In [110]:
# books_second.columns
books_second.head()

,clean_title,clean_author,subjects,source,recleaned_author
0,...And Their Memory Was A Bitter Tree,Robert Howard,"['Fiction, fantasy, short stories']",openlibrary.org,"howard, robert"
1,(Non)Conform Russian And Soviet Art 1958-1995,Barbara Thiemann,[],library of congress,"thiemann, barbara"
2,Scuse Me While I Kiss The Sky,David Henderson,[],library of congress,"henderson, david"
3,A Black Gaze,Tina Campt,[],library of congress,"henderson, david"
4,A Black Rose Thrived,Rochelle Richey,[],library of congress,"campt, tina"


Index(['clean_title', 'clean_author', 'subjects', 'source'], dtype='object')

In [103]:
# clean data
books_second["recleaned_author"] = (
    data["recleaned_author"]
    .str.lower()
)

books_second["recleaned_author"].head()

0       howard, robert
1    thiemann, barbara
2     henderson, david
3     henderson, david
4          campt, tina
Name: recleaned_author, dtype: object

In [104]:
books_second.columns

Index(['clean_title', 'clean_author', 'subjects', 'source',
       'recleaned_author'],
      dtype='object')

In [105]:
books_second.head(20)

,clean_title,clean_author,subjects,source,recleaned_author
0,...And Their Memory Was A Bitter Tree,Robert Howard,"['Fiction, fantasy, short stories']",openlibrary.org,"howard, robert"
1,(Non)Conform Russian And Soviet Art 1958-1995,Barbara Thiemann,[],library of congress,"thiemann, barbara"
2,Scuse Me While I Kiss The Sky,David Henderson,[],library of congress,"henderson, david"
3,A Black Gaze,Tina Campt,[],library of congress,"henderson, david"
4,A Black Rose Thrived,Rochelle Richey,[],library of congress,"campt, tina"
5,A Brief History Of Album Covers,Jason Draper,"['Sound recordings', 'Packaging']",openlibrary.org,"richey, rochelle"
6,A Brief History Of Manga,Helen Mccarthy,[],library of congress,"draper, jason"
7,A Brief History Of Vice,Robert Evans,"['Vice', 'Modern Civilization', 'History', 'Ci...",openlibrary.org,"mccarthy, helen"
8,A Celebration Of Childhood,Helen Sudell,[],library of congress,"evans, robert"
9,A Centaur'S Life 10,Kei Murayama,"['Centaurs', 'Comic books, strips', 'High scho...",openlibrary.org,"sudell, helen"


In [108]:
# collect data from loc and save to .csv file
# books["subjects"] = None
# books["source"] = None
for i, row in books_second.iterrows():
        
    # if subject value is not [], continue
    if row["subjects"] != "[]":
        continue
        
    subj = get_subject(
        row["clean_title"],
        row["recleaned_author"]
    )

    books_second.at[i, "subjects"] = json.dumps(subj)
    books_second.at[i, "source"] = identify_second_source(subj)

    if i % 500 == 0: # save every 500 books
        books_second.to_csv(
            "library_subjects_match.csv",
            index=False
        )

    if i % 100 == 0:
        print(f"{i} completed")

    time.sleep(2) 

# final save
books_second.to_csv(
    "library_subjects_match.csv",
    index=False
)

print("finished loading all data")

100 completed
200 completed
300 completed
400 completed
700 completed
900 completed
1000 completed
1100 completed


KeyboardInterrupt: 

In [109]:
books_second.to_csv(
            "library_subjects_match.csv",
            index=False
        )

In [ ]:
data = data.merge(
    books[
        [
            "clean_title",
            "clean_author",
            "subjects",
            "source"
        ]
    ],
    on=["clean_title", "clean_author"],
    how="left"
)

books.head()

In [37]:
books = books[books["subjects"].apply(lambda x: x != [])]

In [38]:
books.shape

(3792, 4)

## Join datasets by title

### Are there differences between unit_reason and reason?

In [24]:
## step 1: convert to lowercase
data["clean_unit_reason"] = (
    data["unit_reason"]
    .str.lower()
)

# data["clean_unit_reason"].head()

In [25]:
## step 2: remove punctuation
data["clean_unit_reason"] = (
    data["clean_unit_reason"]
    .str.translate(
        str.maketrans("", "", string.punctuation)
    )
)

# data["clean_unit_reason"].head()

In [26]:
## step 3: remove 'pgs' and numbers
data["clean_unit_reason"] = (
    data["clean_unit_reason"]
    .str.replace(r"\bpgs\b", "", regex = True) # remove pgs
    .str.replace(r"\bpg\b", "", regex = True) # remove pg
    .str.replace(r"\bpage\b", "", regex = True) # remove page
    .str.replace(r"\bpages\b", "", regex = True) # remove pages
    .str.replace(r"\d+", "", regex = True) # remove numbers
    .str.strip()
)

# data["clean_unit_reason"].head(100)

In [18]:
## step 4: remove stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)

In [27]:
data["clean_unit_reason"] = data["clean_unit_reason"].apply( # have column
    lambda x: " ".join(
        word for word in x.split()
        if word not in stopwords
    ) if isinstance(x,str) else "" # if data is not string, replace with string
)

# data["clean_unit_reason"].head(10)

## Queries

In [20]:
# find most common words in reason column
all_words = " ".join(data["clean_reason"].dropna()).lower().split()

Counter(all_words).most_common(50)

[('sexually', 3906),
 ('explicit', 3868),
 ('images', 3290),
 ('rape', 1355),
 ('contain', 1116),
 ('contains', 852),
 ('child', 783),
 ('image', 665),
 ('nude', 597),
 ('sei', 471),
 ('techniques', 469),
 ('sex', 395),
 ('age', 364),
 ('fighting', 352),
 ('entire', 344),
 ('book', 336),
 ('wminor', 278),
 ('incest', 240),
 ('minor', 229),
 ('escape', 206),
 ('offdef', 200),
 ('ind', 194),
 ('facilitate', 186),
 ('mfg', 176),
 ('detailed', 168),
 ('indecency', 166),
 ('information', 166),
 ('wchild', 160),
 ('photo', 143),
 ('texas', 143),
 ('graphic', 134),
 ('depiction', 131),
 ('wpn', 125),
 ('security', 124),
 ('manipulation', 111),
 ('maps', 104),
 ('cover', 102),
 ('manufacture', 96),
 ('insert', 92),
 ('brk', 86),
 ('ftrdtr', 85),
 ('offensivedefensive', 85),
 ('used', 82),
 ('concerns', 79),
 ('drugs', 73),
 ('weapon', 71),
 ('bst', 68),
 ('brosis', 62),
 ('ages', 62),
 ('chl', 59)]

In [21]:
# find most common words in unit_reason column
all_words = " ".join(data["clean_unit_reason"].dropna()).lower().split()

Counter(all_words).most_common(25)

[('sexually', 3825),
 ('explicit', 3796),
 ('images', 3252),
 ('contains', 1610),
 ('contain', 1571),
 ('rape', 1461),
 ('ages', 1081),
 ('age', 896),
 ('child', 680),
 ('book', 584),
 ('image', 581),
 ('minor', 550),
 ('sex', 545),
 ('sei', 504),
 ('f', 493),
 ('nude', 459),
 ('reason', 449),
 ('entire', 412),
 ('techniques', 391),
 ('g', 274),
 ('fighting', 273),
 ('incest', 253),
 ('gs', 233),
 ('information', 190),
 ('graphic', 161)]

### Let's use N-grams

In [22]:
from nltk import ngrams

In [23]:
# all_words is from the reason column
bigrams = list(
    ngrams(all_words, 2)
)

Counter(bigrams).most_common(25)

[(('sexually', 'explicit'), 3681),
 (('explicit', 'images'), 3103),
 (('contain', 'sexually'), 898),
 (('images', 'sexually'), 783),
 (('contains', 'sexually'), 641),
 (('explicit', 'image'), 551),
 (('nude', 'child'), 403),
 (('entire', 'book'), 398),
 (('ages', 'contain'), 374),
 (('images', 'ages'), 364),
 (('reason', 'f'), 364),
 (('images', 'contain'), 307),
 (('sex', 'minor'), 286),
 (('book', 'contains'), 242),
 (('fighting', 'techniques'), 239),
 (('age', 'contains'), 236),
 (('minor', 'age'), 232),
 (('ages', 'sexually'), 229),
 (('rape', 'minor'), 204),
 (('images', 'contains'), 190),
 (('rape', 'rape'), 185),
 (('contains', 'rape'), 177),
 (('images', 'rape'), 177),
 (('ages', 'contains'), 160),
 (('rape', 'sexually'), 150)]

# Conclusions
## Data Cleaning
### After cleaning the data and filtering the most common informative words, recurring reasons for banning a book in Texas state prisons are as follows: 
### 1. sexually explicit images
### 2. nude child & sex wminor
### 3. rape
### 4. incest
### 5. fighting techniques
### 6. ????

In [24]:
data.shape

(9396, 11)

In [25]:
# sum common reasons
## sexually explicit images or sei
data["clean_reason"].str.contains(
    r"sei|sexually explicit images",
    case=False,
    na=False
).sum()

np.int64(3693)

## Filtered Copy of Data

In [28]:
filtered = data.copy()

#3693
filtered = filtered.loc[
    ~filtered["clean_reason"].str.contains(
        r"sei|sexually explicit images|rape",
        case=False,
        na=False
    )
]

#1346
#filtered["clean_reason"].str.contains(
   # "rape",
   # case=False,
   # na=False
#).sum()

np.int64(1346)

In [87]:
## nude child
data["clean_reason"].str.contains(
    r"nude child|nud chld",
    case=False,
    na=False
).sum()

np.int64(591)

In [86]:
## offdef or offensivedefensive
data["clean_reason"].str.contains(
    r"fighting techniques|offdef|offensivedefensive",
    case=False,
    na=False
).sum()

np.int64(363)

In [80]:
## sex wminor or sex minor
data["clean_reason"].str.contains(
    r"sex wminor|sex minor",
    case=False,
    na=False
).sum()

np.int64(354)

In [85]:
## ind wchild or indecency child
data["clean_reason"].str.contains(
    r"ind wchild|indecency child",
    case=False,
    na=False
).sum()

np.int64(295)

In [65]:
## incest
data["clean_reason"].str.contains("incest", case=False, na=False).sum()

np.int64(229)

In [74]:
## escape
data["clean_reason"].str.contains("escape", case=False, na=False).sum()

np.int64(206)

In [84]:
## security
data["clean_reason"].str.contains("security", case=False, na=False).sum()

np.int64(120)

## Output Queries

In [36]:
data[data["reason"].str.contains("security concern", case=False, na=False)]

,publication,author,date,reason,year,month,day,state
102,A MESSAGE TO EVERY PRISONER IN AMERICA,"JONES, BRYACE",2022-08-30,ENTIRE BOOK CONTAINS INFORMATION DEEMED A SECU...,2022,8,30,tx
111,A PANTHER IS A BLACK CAT,"MAJOR, REGINALD",2015-01-14,ENTIRE BOOK CONTAINS INFORMATION THAT COULD CA...,2015,1,14,tx
316,ALL IN ONE COMPTIA NETWORK + EXAM N10-005,"MEYERS, MIKE",2022-03-03,ENTIRE PUBLICATION CONTAINS SECURITY CONCERNS,2022,3,3,tx
328,ALL-IN-ONE COMPTIA NETWORK + CERTIFICATION EXAM,"MEYERS, MIKE",2022-03-28,ENTIRE BOOK CONTAINS INFORMATION DEEMED A SECU...,2022,3,28,tx
406,AN AUTHENTIC GUIDE TO KALI LINUX,"BHOOPLI, SAMARAT",2020-02-03,"PAGES 10, 11, 50, 61 & 62 CONTAIN INFORMATION ...",2020,2,3,tx
...,...,...,...,...,...,...,...,...
7840,SUMMARY THE LAWS OF HUMAN NATURE,"GREENE, ROBERT",2020-01-03,ENTIRE BOOK CONTAINS MANIPULATION TECHNIQUES W...,2020,1,3,tx
8860,WHEN HELL BECOMES YOUR HOME,"COGNITO, N.",2022-03-31,ENTIRE BOOK CONTAINS SECURITY CONCERNS,2022,3,31,tx
8969,WINDOWS KERNAL PROGRAMMING,"YOSIFOVICH, PAVEL",2021-12-15,ENTIRE BOOK CONTAINS SECURITY CONCERNS,2021,12,15,tx
8990,WIRESHARK FOR SECURITY PROFESSIONALS,"BULLOCK, JESSEY",2022-07-28,ENTIRE BOOK CONTAINS INFORMATION DEEMED A SECU...,2022,7,28,tx


## Join Google Books Data w/ Banned Books Data